# Machine Learning - Assignment 2

**Name:** Praveen Garg<br>
**BITS ID:** 2026AJ05478

## Predicting Online Shoppers' Purchasing Intention

This notebook uses the Online Shoppers Purchasing Intention dataset from the UCI Machine
Learning Repository to predict whether a browsing session on a shopping site ends in a
purchase (`Revenue = True`). It loads and checks the data, preprocesses it, trains and
evaluates five classification models, and compares them. It also writes
`test_data.csv` for the Streamlit app and saves the five trained models in `model/`.

In [1]:
# =============================================================================
# Libraries and Global Configuration
# =============================================================================

import os
import warnings

import joblib
import pandas as pd
import numpy as np
import sklearn

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    OneHotEncoder,
    StandardScaler,
)
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold,
    cross_val_score,
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
)

# Define a single fixed random seed used throughout this notebook.
# Fixing it makes the results reproducible, so the final run on the BITS
# Virtual Lab should give the same numbers as my local runs, as long as
# the library versions are the same on both.
RANDOM_STATE = 42

---
## 1. Problem statement

The goal is to predict, for one visit to an online shop, whether the session ends in a
purchase. Each session is described by 17 features. Ten are numeric: how many
administrative, informational and product-related pages the visitor opened and how long
they spent on each kind, three Google Analytics measures of those pages (bounce rate,
exit rate and page value), and how close the visit was to a special day. The other seven
are categorical and describe the visit: the month, operating system, browser, region,
traffic type, visitor type, and whether it fell on a weekend.

This is a binary classification problem. The positive class is a purchase
(`Revenue = True`) and the negative class is no purchase (`Revenue = False`). A shop
would want this prediction while a visitor is still on the site, so it can act on the
sessions that look unlikely to convert, for example with a discount, a live chat prompt
or product recommendations, without handing discounts to people who were going to buy
anyway.

---
## 2. The dataset

I use the Online Shoppers Purchasing Intention Dataset from the
[UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/468/online+shoppers+purchasing+intention+dataset).
It is published under the
[CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) licence, which requires the
source to be credited:

- Creators: C. Sakar and Y. Kastro
- DOI: [10.24432/C5F88Q](https://doi.org/10.24432/C5F88Q)
- Paper: Sakar, C.O., Polat, S.O., Katircioglu, M. and Kastro, Y. (2019). *Real-time
  prediction of online shoppers' purchasing intention using multilayer perceptron and
  LSTM recurrent neural networks*. Neural Computing and Applications 31, 6893-6908.

Each row is one browsing session on an e-commerce site. According to the creators, the
data was put together so that each session belongs to a different user over a one-year
period, so that it does not lean towards any particular campaign, special day, user
profile or period.

In [2]:
# =============================================================================
# Load and Inspect Dataset
# =============================================================================

# Load the online shoppers intention dataset from the data directory
csv_file_path = 'data/online_shoppers_intention.csv'
shoppers_df = pd.read_csv(csv_file_path)

# 1. Dataset dimensions
print("=== Dataset Shape ===")
print(f"Shape: {shoppers_df.shape}\n")

# 2. Column data types
print("=== Column Data Types ===")
print(shoppers_df.dtypes)
print()

# 3. Missing values per column
print("=== Missing Values per Column ===")
missing_per_col = shoppers_df.isna().sum()
print(missing_per_col)
print(f"Total missing values: {missing_per_col.sum()}\n")

# 4. Duplicate rows and positive-revenue duplicates
duplicate_rows_mask = shoppers_df.duplicated()
duplicate_rows_count = duplicate_rows_mask.sum()
true_revenue_duplicates = (
    duplicate_rows_mask & shoppers_df['Revenue']
).sum()
print("=== Duplicate Rows Analysis ===")
print(f"Duplicate rows: {duplicate_rows_count}")
print(f"Duplicates with Revenue True: {true_revenue_duplicates}\n")

# 5. Target counts and share
revenue_counts = shoppers_df['Revenue'].value_counts()
revenue_shares = shoppers_df['Revenue'].value_counts(normalize=True)
print("=== Target Distribution (Revenue) ===")
print(
    f"Revenue False: {revenue_counts[False]}, "
    f"Revenue True: {revenue_counts[True]}"
)
print(
    f"True share: {revenue_shares[True]:.3f} "
    f"({revenue_shares[True] * 100:.1f}%)"
)
print(
    f"False share: {revenue_shares[False]:.3f} "
    f"({revenue_shares[False] * 100:.1f}%)"
)

=== Dataset Shape ===
Shape: (12330, 18)

=== Column Data Types ===
Administrative               int64
Administrative_Duration    float64
Informational                int64
Informational_Duration     float64
ProductRelated               int64
ProductRelated_Duration    float64
BounceRates                float64
ExitRates                  float64
PageValues                 float64
SpecialDay                 float64
Month                          str
OperatingSystems             int64
Browser                      int64
Region                       int64
TrafficType                  int64
VisitorType                    str
Weekend                       bool
Revenue                       bool
dtype: object

=== Missing Values per Column ===
Administrative             0
Administrative_Duration    0
Informational              0
Informational_Duration     0
ProductRelated             0
ProductRelated_Duration    0
BounceRates                0
ExitRates                  0
PageValues           

In [3]:
# =============================================================================
# Features and Target Separation
# =============================================================================

# Split the 17 input features (X) from the target (y).
# The duplicates stay in for now; I deal with them in the preprocessing step.
X = shoppers_df.drop(columns=['Revenue'])
y = shoppers_df['Revenue']

# Print shapes of the feature matrix and target vector
print("=== Feature Matrix and Target Vector Shapes ===")
print(f"Feature matrix shape (X): {X.shape}")
print(f"Target vector shape (y): {y.shape}\n")

# Group the features by type: 10 numeric, 7 categorical (4 of them coded)
numeric_columns = [
    'Administrative', 'Administrative_Duration', 'Informational',
    'Informational_Duration', 'ProductRelated', 'ProductRelated_Duration',
    'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay'
]
categorical_columns = [
    'Month', 'OperatingSystems', 'Browser', 'Region',
    'TrafficType', 'VisitorType', 'Weekend'
]
coded_categorical_columns = [
    'OperatingSystems', 'Browser', 'Region', 'TrafficType'
]

# Make sure the two lists together cover every feature exactly once,
# so no column gets missed or used twice later on
assert sorted(numeric_columns + categorical_columns) == sorted(X.columns)

print("=== Feature Type Breakdown ===")
print(f"Total features: {X.shape[1]}")
print(f"Numeric features: {len(numeric_columns)}")
print(f"Categorical features: {len(categorical_columns)}")
print(
    "Categorical features stored as number codes: "
    f"{len(coded_categorical_columns)}"
)

# Number of distinct values in each categorical column
print("\n=== Distinct Values per Categorical Column ===")
print(X[categorical_columns].nunique())

=== Feature Matrix and Target Vector Shapes ===
Feature matrix shape (X): (12330, 17)
Target vector shape (y): (12330,)

=== Feature Type Breakdown ===
Total features: 17
Numeric features: 10
Categorical features: 7
Categorical features stored as number codes: 4

=== Distinct Values per Categorical Column ===
Month               10
OperatingSystems     8
Browser             13
Region               9
TrafficType         20
VisitorType          3
Weekend              2
dtype: int64


**Dataset description**

The dataset has 12,330 rows, one per session, and 18 columns. Taking out the target,
`Revenue`, leaves 17 input features, which is well above the minimum of 12 features and
500 instances that the assignment asks for.

Ten of the features are numeric. Six describe the browsing itself: the number of
administrative, informational and product-related pages visited, and the time spent on
each of the three kinds. The other four are the Google Analytics page measures
(`BounceRates`, `ExitRates`, `PageValues`) and `SpecialDay`, which says how close the
visit was to a special day such as Valentine's Day.

The remaining seven are categorical. `Month` (10 distinct months) and `VisitorType`
(`Returning_Visitor`, `New_Visitor` or `Other`) are stored as text, and `Weekend` as
True/False. `OperatingSystems`, `Browser`, `Region` and `TrafficType` are stored as
integers, with 8, 13, 9 and 20 distinct codes, but the numbers are only labels for a
particular operating system, browser, region or traffic source. Browser 4 is not "more"
than browser 2, so these four have to be treated as categories and not as quantities.

The target `Revenue` is True/False, and the positive class is True, a purchase. The
classes are imbalanced: 10,422 sessions (84.5%) ended without a purchase and only 1,908
(15.5%) ended with one.

There are no missing values in any of the 18 columns. There are 125 duplicate rows,
meaning exact copies of an earlier row, and all 125 have `Revenue = False`.

The imbalance changes how I judge the models. A model that predicts "no purchase" for
every session would already get 84.5% accuracy while finding none of the buyers, so its
recall, F1 and MCC would all be 0. Accuracy on its own is misleading here. I still
report it, since the assignment asks for all six metrics, but I compare the models
mainly on MCC, with F1 and recall as support. All three only improve when buyers are
actually found.

---
## 3. Preprocessing

The 125 duplicate rows are all sessions that bounced immediately: one or two
product pages, a total time of 0 across all three duration columns, bounce and
exit rates of exactly 0.2 (the highest value either column takes), and a page
value of 0. The UCI description says each session belongs to a different
user, so these are most likely separate real visits that happen to produce
identical rows, not data-entry errors. I keep them.

I split the 10 numeric features into two groups based on skew. Seven columns
are heavily right-skewed (skew 1.96 to 7.58), so I apply log1p to compress the
long tail and then StandardScaler to centre and unit-scale them. I use log1p
rather than log because all seven have a minimum of 0, and log(0) is
undefined. The other three (BounceRates, ExitRates, SpecialDay) are capped at
small values (0.2 for the two rates, 1 for SpecialDay), so there are no
extreme values to pull in, and I only scale them.

The 7 categorical columns are one-hot encoded. Using `handle_unknown='ignore'`
means the encoder outputs all-zeros for any unseen category code at prediction
time, which protects the app from crashing on input it has not seen before.

Logistic regression, k-NN and SVM all use distance or gradient information and
are sensitive to feature scale, so they need the scaler. Decision trees split
on individual thresholds and are unaffected by scale. Gaussian Naive Bayes fits
a Gaussian on each feature independently and also does not need scaling,
although keeping it scaled does no harm.

In [4]:
# =============================================================================
# 3a. Inspect Duplicate Rows
# =============================================================================

# Pull out only the rows that are exact copies of an earlier row
dup_mask = shoppers_df.duplicated()
dup_df = shoppers_df[dup_mask]

# How many product-related pages did those sessions visit?
print("=== ProductRelated page counts in duplicates ===")
print(dup_df['ProductRelated'].value_counts())

# Did those sessions spend any time on pages at all?
dur_cols = [
    'Administrative_Duration',
    'Informational_Duration',
    'ProductRelated_Duration',
]
print("\n=== Duration column totals for duplicates ===")
print(dup_df[dur_cols].sum())

# Check the bounce/exit rates and page value for duplicates
print("\n=== BounceRates unique values in duplicates ===")
print(dup_df['BounceRates'].unique())
print("=== ExitRates unique values in duplicates ===")
print(dup_df['ExitRates'].unique())
print("=== PageValues unique values in duplicates ===")
print(dup_df['PageValues'].unique())

# Decision: keep the duplicate rows.
# All 125 are pure bounce sessions (one or two product pages, zero time on
# any page, bounce and exit rates of 0.2, zero page value). The UCI
# description says each session is a different user, so these are plausible
# real visits, and dropping them would under-count this kind of visit.

=== ProductRelated page counts in duplicates ===
ProductRelated
1    118
2      7
Name: count, dtype: int64

=== Duration column totals for duplicates ===
Administrative_Duration    0.0
Informational_Duration     0.0
ProductRelated_Duration    0.0
dtype: float64

=== BounceRates unique values in duplicates ===
[0.2]
=== ExitRates unique values in duplicates ===
[0.2]
=== PageValues unique values in duplicates ===
[0.]


In [5]:
# =============================================================================
# 3b. Skew Analysis of Numeric Columns
# =============================================================================

# Split the 10 numeric features into two groups:
#   skewed_cols  - heavily right-skewed, need log1p before scaling
#   bounded_cols - capped at small values, scaled directly
skewed_cols = [
    'Administrative',
    'Administrative_Duration',
    'Informational',
    'Informational_Duration',
    'ProductRelated',
    'ProductRelated_Duration',
    'PageValues',
]
bounded_cols = ['BounceRates', 'ExitRates', 'SpecialDay']

# Confirm the two groups together cover numeric_columns exactly
assert sorted(skewed_cols + bounded_cols) == sorted(numeric_columns), (
    "skewed_cols and bounded_cols must together equal numeric_columns"
)

# Print a compact table: original skew and skew after log1p transform
header = f"{'column':<30} {'min':>5} {'skew':>7} {'skew log1p':>12}"
print(header)
print("-" * len(header))
for col in skewed_cols:
    mn = shoppers_df[col].min()
    sk = shoppers_df[col].skew()
    sk_log = np.log1p(shoppers_df[col]).skew()
    print(f"{col:<30} {mn:>5.0f} {sk:>7.2f} {sk_log:>12.2f}")

# The bounded columns have no long tail to compress: show their range
print("\n=== Range of the bounded columns ===")
print(shoppers_df[bounded_cols].agg(['min', 'max']))

column                           min    skew   skew log1p
---------------------------------------------------------
Administrative                     0    1.96         0.57
Administrative_Duration            0    5.62         0.26
Informational                      0    4.04         2.03
Informational_Duration             0    7.58         1.92
ProductRelated                     0    4.34        -0.05
ProductRelated_Duration            0    7.26        -1.42
PageValues                         0    6.38         1.81

=== Range of the bounded columns ===
     BounceRates  ExitRates  SpecialDay
min          0.0        0.0         0.0
max          0.2        0.2         1.0


In [6]:
# =============================================================================
# 3c. Preprocessor Factory
# =============================================================================

# This is a function, not a single ColumnTransformer object, because each
# model in the next section needs its own fitted preprocessor inside its own
# Pipeline. If the same unfitted transformer object were reused, the second
# call to .fit() would overwrite the first, and each model's pipeline would
# share the same internal state.


def make_preprocessor():
    """Return a fresh, unfitted ColumnTransformer for this dataset."""
    # For the skewed columns: apply log1p first (using np.log1p directly,
    # not a lambda - lambdas cannot be pickled when saving the model),
    # then standardise.
    log_and_scale = Pipeline([
        ('log1p', FunctionTransformer(np.log1p)),
        ('scaler', StandardScaler()),
    ])

    return ColumnTransformer(
        transformers=[
            ('skewed', log_and_scale, skewed_cols),
            ('bounded', StandardScaler(), bounded_cols),
            # sparse_output=False keeps Gaussian NB from crashing;
            # handle_unknown='ignore' stops unseen codes crashing the app
            (
                'cat',
                OneHotEncoder(
                    handle_unknown='ignore',
                    sparse_output=False,
                ),
                categorical_columns,
            ),
        ]
    )

---
## 4. Train/test split and test_data.csv

I use one 80/20 split, stratified on `Revenue` so both parts keep the same
15.5% buyer rate as the full dataset, and seeded with `RANDOM_STATE` so it
comes out the same on every run. The same split is used for all five
models, so their test results are directly comparable. The test portion is
written to `test_data.csv` as raw, unprocessed data (the 17 feature columns
plus `Revenue`), because the saved models each carry their own fitted
preprocessor and apply it themselves when the app calls them.

In [7]:
# =============================================================================
# 4a. Train / Test Split
# =============================================================================

# One stratified split, seeded, used by every model in later cells
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("=== Split Shapes ===")
print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

print("\n=== Buyer Share ===")
print(f"Train buyer share: {y_train.mean():.4f}")
print(f"Test  buyer share: {y_test.mean():.4f}")
print(f"Buyers in test set: {y_test.sum()}")

=== Split Shapes ===
X_train: (9864, 17)
X_test:  (2466, 17)
y_train: (9864,)
y_test:  (2466,)

=== Buyer Share ===
Train buyer share: 0.1547
Test  buyer share: 0.1549
Buyers in test set: 382


In [8]:
# =============================================================================
# 4b. Write test_data.csv and Verify Preprocessor Output Width
# =============================================================================

# Attach the target back to the raw test features before saving.
# The file stays in raw form so the app can feed it to any saved model,
# and the model's internal preprocessor handles encoding and scaling.
test_export = X_test.copy()
test_export['Revenue'] = y_test.values
test_export.to_csv('test_data.csv', index=False)

# Read it back and confirm it looks right
reloaded = pd.read_csv('test_data.csv')
print("=== test_data.csv shape ===")
print(reloaded.shape)

columns_match = list(reloaded.columns) == list(shoppers_df.columns)
print(f"\nColumn list matches original: {columns_match}")

dtypes_match = (reloaded.dtypes == shoppers_df.dtypes).all()
print(f"Dtypes match original:        {dtypes_match}")

# Fit one preprocessor on training data only (fitting on test data would
# leak test statistics into training), then check the output width
sample_pre = make_preprocessor()
sample_pre.fit(X_train)
train_transformed = sample_pre.transform(X_train)
test_transformed  = sample_pre.transform(X_test)

print("\n=== Preprocessor Output Shapes ===")
print(f"Train transformed: {train_transformed.shape}")
print(f"Test  transformed: {test_transformed.shape}")

=== test_data.csv shape ===
(2466, 18)

Column list matches original: True
Dtypes match original:        True



=== Preprocessor Output Shapes ===
Train transformed: (9864, 75)
Test  transformed: (2466, 75)


---
## 5. Models and evaluation

**The six metrics**

- **Accuracy**: share of all predictions that are correct. Easy to
  read, but misleading when classes are imbalanced (see the dataset
  description in section 2 for why 84.5% is a floor, not a target).
- **AUC**: area under the ROC curve. Measures how well the model ranks
  buyers above non-buyers. Computed from predicted *probabilities* (column
  1 of `predict_proba`, which is the probability of `True`), not from the
  binary predicted labels.
- **Precision**: of the sessions the model calls a purchase, the share
  that really are. High precision means few false alarms.
- **Recall**: of all actual purchases, the share the model finds. High
  recall means few missed buyers.
- **F1**: harmonic mean of precision and recall. A single number that
  captures the trade-off between the two.
- **MCC**: Matthews Correlation Coefficient. Ranges from -1 to +1 and
  accounts for all four cells of the confusion matrix. Because it treats
  both classes the same way, it still gives a fair score when one class
  outnumbers the other by more than 5 to 1, as here.

**How I judge the models**

I use MCC as the main criterion, with F1 and recall as supporting
measures. As the dataset description in section 2 explains, a model that
predicts 'no purchase' for every session gets 84.5% accuracy, so accuracy on
its own cannot tell a useful model from a useless one. MCC, F1 and recall only improve when the
model actually finds buyers.

**How settings are chosen**

Where a model has a setting worth tuning (tree depth, number of neighbours,
Gaussian smoothing), I pick it by 5-fold stratified cross-validation on the
training part only, scored by MCC. The test part is never used to choose a
setting; it only gives the final numbers for each model. Each CV fold fits
its own preprocessor on its own training part, so nothing leaks between
folds.

In [9]:
# =============================================================================
# 5a. Shared Tools
# =============================================================================

# --- Metrics function ------------------------------------------------
# Takes a fitted pipeline, feature set and true labels.
# Returns all six metrics as a plain dict so every model is scored the
# same way. AUC uses predict_proba(:, 1) -- column 1 is True because
# classes_ is [False, True] -- not the predicted binary labels.
def compute_metrics(fitted_pipe, X_features, y_true):
    y_pred = fitted_pipe.predict(X_features)
    y_prob = fitted_pipe.predict_proba(X_features)[:, 1]
    return {
        'Accuracy':  round(accuracy_score(y_true, y_pred), 3),
        'AUC':       round(roc_auc_score(y_true, y_prob), 3),
        'Precision': round(precision_score(y_true, y_pred), 3),
        'Recall':    round(recall_score(y_true, y_pred), 3),
        'F1':        round(f1_score(y_true, y_pred), 3),
        'MCC':       round(matthews_corrcoef(y_true, y_pred), 3),
    }


# --- Printing helper -------------------------------------------------
# Shows the test metrics, confusion breakdown and training metrics
# (the training numbers show whether a model overfits).
def print_results(model_name, fitted_pipe):
    test_m = compute_metrics(fitted_pipe, X_test, y_test)
    train_m = compute_metrics(fitted_pipe, X_train, y_train)
    tn, fp, fn, tp = confusion_matrix(
        y_test, fitted_pipe.predict(X_test)
    ).ravel()
    print(f"=== {model_name} ===")
    print("Test metrics:")
    for name, val in test_m.items():
        print(f"  {name}: {val:.3f}")
    print(f"Confusion (TN / FP / FN / TP): {tn} / {fp} / {fn} / {tp}")
    print(
        f"Train accuracy: {train_m['Accuracy']:.3f}  "
        f"Train MCC: {train_m['MCC']:.3f}"
    )
    return test_m


# --- Pipeline builder ------------------------------------------------
# Wraps any classifier in a Pipeline with a fresh preprocessor.
# Step names are 'prep' and 'model' so grid params use model__<param>.
def build_pipeline(clf):
    return Pipeline([
        ('prep', make_preprocessor()),
        ('model', clf),
    ])


# --- CV splitter -----------------------------------------------------
# Reused by every grid search so all CV runs are comparable.
cv_splitter = StratifiedKFold(
    n_splits=5, shuffle=True, random_state=RANDOM_STATE
)

# --- Result stores ---------------------------------------------------
# test_scores  : {model_name: metrics_dict}
# fitted_pipes : {model_name: fitted_pipeline}
# The comparison table and the saved models both come from these.
test_scores = {}
fitted_pipes = {}

### 5.1 Logistic Regression

Logistic regression with every setting left at its default: the lbfgs
solver, L2 regularisation and `C = 1`. Scaling matters here: lbfgs follows the
gradient, and features on very different scales make it converge slowly, or
not at all within its default 100 iterations. The preprocessor takes care of
that, so the model converges without raising a `ConvergenceWarning`.

In [10]:
# =============================================================================
# 5.1 Logistic Regression
# =============================================================================

# Default LogisticRegression: lbfgs solver, L2 penalty, C=1.
# Scaled inputs matter -- the solver uses gradient steps.
lr_pipeline = build_pipeline(LogisticRegression())
lr_pipeline.fit(X_train, y_train)

lr_test_metrics = print_results('Logistic Regression', lr_pipeline)
test_scores['Logistic Regression'] = lr_test_metrics
fitted_pipes['Logistic Regression'] = lr_pipeline

=== Logistic Regression ===
Test metrics:
  Accuracy: 0.894
  AUC: 0.911
  Precision: 0.705
  Recall: 0.545
  F1: 0.614
  MCC: 0.561
Confusion (TN / FP / FN / TP): 1997 / 87 / 174 / 208
Train accuracy: 0.900  Train MCC: 0.586


### 5.2 Decision Tree

An unlimited decision tree keeps splitting until every leaf is pure, so it
memorises the training data: in the CV below it scores a training MCC of
1.000 but only 0.476 on the held-out folds. Capping `max_depth`
stops it before it starts fitting noise. I chose the depth by
5-fold CV on the training set, scoring by MCC, over the grid
[3, 4, 5, 6, 7, 8, 10, 12, None]. The best value was depth 5
(CV MCC 0.589), well away from both ends of the grid. `random_state`
is fixed so that ties between equally good splits are broken the same way
on every run.

In [11]:
# =============================================================================
# 5.2 Decision Tree
# =============================================================================

# Grid search over max_depth; None means unlimited (included to show
# the gap between constrained and unconstrained fits)
dt_param_grid = {
    'model__max_depth': [3, 4, 5, 6, 7, 8, 10, 12, None],
}
# return_train_score=True also records the MCC on each fold's training
# part, which shows how far the deeper trees overfit
dt_search = GridSearchCV(
    build_pipeline(
        DecisionTreeClassifier(random_state=RANDOM_STATE)
    ),
    param_grid=dt_param_grid,
    scoring='matthews_corrcoef',
    cv=cv_splitter,
    return_train_score=True,
)
dt_search.fit(X_train, y_train)

# Labels come from the grid itself, so they cannot drift from it
print("Decision Tree CV results:")
for depth, train_score, cv_score in zip(
    dt_param_grid['model__max_depth'],
    dt_search.cv_results_['mean_train_score'],
    dt_search.cv_results_['mean_test_score'],
):
    depth_str = str(depth)
    print(
        f"  max_depth={depth_str:<4}  train MCC: {train_score:.3f}"
        f"  mean CV MCC: {cv_score:.3f}"
    )

print(f"Best params: {dt_search.best_params_}")
print(f"Best CV MCC: {dt_search.best_score_:.3f}")

# best_estimator_ is already refitted on the full training set
dt_test_metrics = print_results(
    'Decision Tree', dt_search.best_estimator_
)
test_scores['Decision Tree'] = dt_test_metrics
fitted_pipes['Decision Tree'] = dt_search.best_estimator_

Decision Tree CV results:
  max_depth=3     train MCC: 0.591  mean CV MCC: 0.574
  max_depth=4     train MCC: 0.620  mean CV MCC: 0.586
  max_depth=5     train MCC: 0.644  mean CV MCC: 0.589
  max_depth=6     train MCC: 0.666  mean CV MCC: 0.575
  max_depth=7     train MCC: 0.700  mean CV MCC: 0.576
  max_depth=8     train MCC: 0.741  mean CV MCC: 0.553
  max_depth=10    train MCC: 0.823  mean CV MCC: 0.526
  max_depth=12    train MCC: 0.892  mean CV MCC: 0.505
  max_depth=None  train MCC: 1.000  mean CV MCC: 0.476
Best params: {'model__max_depth': 5}
Best CV MCC: 0.589
=== Decision Tree ===
Test metrics:
  Accuracy: 0.899
  AUC: 0.917
  Precision: 0.711
  Recall: 0.586
  F1: 0.643
  MCC: 0.588
Confusion (TN / FP / FN / TP): 1993 / 91 / 158 / 224
Train accuracy: 0.910  Train MCC: 0.634


### 5.3 K-Nearest Neighbours

k-NN labels a session by a majority vote among the k most similar
sessions in the training data, measured after preprocessing. A small k
follows noise in the training data and a large k blurs real differences,
so k has to be tuned. Scaling is essential here: without it, the columns
with large ranges, such as the durations in seconds, would dominate the
distance and drown out the rest. I chose k by the same 5-fold CV, over the
grid
[3, 5, 7, 9, 11, 15, 21, 31, 41, 51]. The best was k = 15 (CV MCC
0.568); k = 31 tied it, so either would have been a reasonable choice.

In [12]:
# =============================================================================
# 5.3 K-Nearest Neighbours
# =============================================================================

knn_param_grid = {
    'model__n_neighbors': [3, 5, 7, 9, 11, 15, 21, 31, 41, 51],
}
knn_search = GridSearchCV(
    build_pipeline(KNeighborsClassifier()),
    param_grid=knn_param_grid,
    scoring='matthews_corrcoef',
    cv=cv_splitter,
)
knn_search.fit(X_train, y_train)

print("KNN CV results:")
for k, cv_score in zip(
    knn_param_grid['model__n_neighbors'],
    knn_search.cv_results_['mean_test_score'],
):
    print(f"  n_neighbors={k:<3}  mean CV MCC: {cv_score:.3f}")

print(f"Best params: {knn_search.best_params_}")
print(f"Best CV MCC: {knn_search.best_score_:.3f}")

knn_test_metrics = print_results('KNN', knn_search.best_estimator_)
test_scores['KNN'] = knn_test_metrics
fitted_pipes['KNN'] = knn_search.best_estimator_

KNN CV results:
  n_neighbors=3    mean CV MCC: 0.524
  n_neighbors=5    mean CV MCC: 0.547
  n_neighbors=7    mean CV MCC: 0.553
  n_neighbors=9    mean CV MCC: 0.554
  n_neighbors=11   mean CV MCC: 0.552
  n_neighbors=15   mean CV MCC: 0.568
  n_neighbors=21   mean CV MCC: 0.565
  n_neighbors=31   mean CV MCC: 0.568
  n_neighbors=41   mean CV MCC: 0.564
  n_neighbors=51   mean CV MCC: 0.560
Best params: {'model__n_neighbors': 15}
Best CV MCC: 0.568


=== KNN ===
Test metrics:
  Accuracy: 0.894
  AUC: 0.896
  Precision: 0.705
  Recall: 0.545
  F1: 0.614
  MCC: 0.561
Confusion (TN / FP / FN / TP): 1997 / 87 / 174 / 208
Train accuracy: 0.906  Train MCC: 0.614


### 5.4 Naive Bayes

I use Gaussian Naive Bayes, not Multinomial. Multinomial NB requires
non-negative integer-like inputs (it was designed for word counts), but
after log1p and StandardScaler the numeric features go negative. Gaussian
NB models each feature within each class as a Gaussian distribution, which
suits continuous and scaled data.

With the default settings it goes badly wrong. Gaussian NB fits a Gaussian
for every feature-class pair. A few rare one-hot codes never occur among the
buyers in the training part, so for the buyer class those columns are always
0 and their variance is zero (the cell below counts 5 of them). The default
`var_smoothing` adds only a tiny amount to that, so the buyer-class Gaussian
for each of these columns is an extremely narrow, tall spike at 0. Almost
every session has a 0 in those columns, so it lands right on the spike and
gets an enormous likelihood under the buyer class. The model then puts almost
every session into the buyer class: 2,149 of 2,466 test sessions (87%) are
predicted as buyers, and accuracy drops to 0.275.

`var_smoothing` is the fix. It adds a share of the
largest feature variance in the training data to every per-class
variance. It prevents any variance from reaching zero and damps the
spikes. I chose it by 5-fold CV (same scoring, same splitter), over
[1e-9, 1e-3, 0.1, 0.3, 0.5, 1, 2, 3, 5, 10]. The best value was 0.5
(CV MCC 0.519), comfortably inside the grid.

In [13]:
# =============================================================================
# 5.4 Naive Bayes (Gaussian)
# =============================================================================

# --- Default GaussianNB: show what goes wrong ----------------------
default_gnb_pipe = build_pipeline(GaussianNB())
default_gnb_pipe.fit(X_train, y_train)
default_preds = default_gnb_pipe.predict(X_test)
default_buyer_count = int(default_preds.sum())
print("Default GaussianNB:")
buyer_share = default_buyer_count / len(default_preds)
print(
    f"  Predicted as buyers: {default_buyer_count} of {len(default_preds)}"
    f" ({buyer_share:.0%})"
)
default_m = compute_metrics(default_gnb_pipe, X_test, y_test)
print(f"  Accuracy: {default_m['Accuracy']:.3f}")
print(f"  MCC:      {default_m['MCC']:.3f}")

# --- Why: zero-variance columns in the buyer class -----------------
# Fit a standalone preprocessor to inspect transformed training buyers
diagnostic_pre = make_preprocessor()
X_train_arr = diagnostic_pre.fit_transform(X_train)
buyer_rows = X_train_arr[y_train.values]
zero_var_count = int((buyer_rows.var(axis=0) == 0).sum())
print(
    "\nZero-variance columns in buyer class"
    f" after preprocessing: {zero_var_count}"
)
print(
    "These are rare one-hot codes that never appear"
    " among training buyers."
)

# --- Tuned GaussianNB: grid search over var_smoothing --------------
gnb_param_grid = {
    'model__var_smoothing': [
        1e-9, 1e-3, 0.1, 0.3, 0.5, 1, 2, 3, 5, 10
    ],
}
gnb_search = GridSearchCV(
    build_pipeline(GaussianNB()),
    param_grid=gnb_param_grid,
    scoring='matthews_corrcoef',
    cv=cv_splitter,
)
gnb_search.fit(X_train, y_train)

print("\nGaussianNB CV results:")
for vs, cv_score in zip(
    gnb_param_grid['model__var_smoothing'],
    gnb_search.cv_results_['mean_test_score'],
):
    vs_str = str(vs)
    print(f"  var_smoothing={vs_str:<6}  mean CV MCC: {cv_score:.3f}")

print(f"Best params: {gnb_search.best_params_}")
print(f"Best CV MCC: {gnb_search.best_score_:.3f}")

gnb_test_metrics = print_results(
    'Naive Bayes', gnb_search.best_estimator_
)
test_scores['Naive Bayes'] = gnb_test_metrics
fitted_pipes['Naive Bayes'] = gnb_search.best_estimator_

Default GaussianNB:
  Predicted as buyers: 2149 of 2466 (87%)
  Accuracy: 0.275
  MCC:      0.131

Zero-variance columns in buyer class after preprocessing: 5
These are rare one-hot codes that never appear among training buyers.



GaussianNB CV results:
  var_smoothing=1e-09   mean CV MCC: 0.152
  var_smoothing=0.001   mean CV MCC: 0.326
  var_smoothing=0.1     mean CV MCC: 0.446
  var_smoothing=0.3     mean CV MCC: 0.494
  var_smoothing=0.5     mean CV MCC: 0.519
  var_smoothing=1       mean CV MCC: 0.516
  var_smoothing=2       mean CV MCC: 0.444
  var_smoothing=3       mean CV MCC: 0.282
  var_smoothing=5       mean CV MCC: 0.027
  var_smoothing=10      mean CV MCC: 0.000
Best params: {'model__var_smoothing': 0.5}
Best CV MCC: 0.519
=== Naive Bayes ===
Test metrics:
  Accuracy: 0.821
  AUC: 0.841
  Precision: 0.454
  Recall: 0.762
  F1: 0.569
  MCC: 0.490
Confusion (TN / FP / FN / TP): 1734 / 350 / 91 / 291
Train accuracy: 0.828  Train MCC: 0.520


### 5.5 Support Vector Machine

An RBF-kernel SVM with the defaults, `C = 1` and `gamma = 'scale'`. The RBF
kernel lets the SVM draw a curved boundary between buyers and non-buyers,
instead of the straight one logistic regression is limited to. It works on
distances between sessions, so scaling is critical here too.

`probability=True` makes the SVC run an internal 5-fold cross-calibration
(Platt scaling) so it can produce `predict_proba` outputs for AUC. Without
it, `predict_proba` is unavailable and the metrics function would fail. The
app also calls `predict_proba` on every model, so this keeps all five
models interchangeable there.

The model is not grid-searched. Each fit is much slower than the other
models' fits, because Platt scaling fits five extra models internally, so
tuning C and gamma over even a small grid would take several minutes. The
defaults already score close to logistic regression on this dataset.

In [14]:
# =============================================================================
# 5.5 Support Vector Machine (RBF kernel)
# =============================================================================

# probability=True enables predict_proba via internal Platt scaling.
# In sklearn 1.9 this triggers a FutureWarning recommending
# CalibratedClassifierCV instead. I silence it here: the parameter still
# works until sklearn 1.11, and requirements.txt pins the version the
# models were trained with.
with warnings.catch_warnings():
    warnings.simplefilter('ignore', FutureWarning)
    svm_pipeline = build_pipeline(
        SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE)
    )
    svm_pipeline.fit(X_train, y_train)

svm_test_metrics = print_results(
    'Support Vector Machine', svm_pipeline
)
test_scores['Support Vector Machine'] = svm_test_metrics
fitted_pipes['Support Vector Machine'] = svm_pipeline

=== Support Vector Machine ===
Test metrics:
  Accuracy: 0.894
  AUC: 0.900
  Precision: 0.700
  Recall: 0.550
  F1: 0.616
  MCC: 0.561
Confusion (TN / FP / FN / TP): 1994 / 90 / 172 / 210
Train accuracy: 0.916  Train MCC: 0.658


---
## 6. Comparison table

These are the test-set results of all five models on the same split, and
they are the numbers that go into the README comparison table.

In [15]:
# =============================================================================
# 6a. Comparison Table
# =============================================================================

# One row per model, in the same order as the README table, and one
# column per metric, taken straight from the test-set scores collected
# while fitting each model above.
readme_model_order = [
    'Logistic Regression',
    'Decision Tree',
    'KNN',
    'Naive Bayes',
    'Support Vector Machine',
]
comparison_table = pd.DataFrame(test_scores).T.loc[readme_model_order]

print("=== Test-Set Comparison Table ===")
print(comparison_table)

print("\n=== Best Model per Metric ===")
print(comparison_table.idxmax())

=== Test-Set Comparison Table ===
                        Accuracy    AUC  Precision  Recall     F1    MCC
Logistic Regression        0.894  0.911      0.705   0.545  0.614  0.561
Decision Tree              0.899  0.917      0.711   0.586  0.643  0.588
KNN                        0.894  0.896      0.705   0.545  0.614  0.561
Naive Bayes                0.821  0.841      0.454   0.762  0.569  0.490
Support Vector Machine     0.894  0.900      0.700   0.550  0.616  0.561

=== Best Model per Metric ===
Accuracy     Decision Tree
AUC          Decision Tree
Precision    Decision Tree
Recall         Naive Bayes
F1           Decision Tree
MCC          Decision Tree
dtype: str


In [16]:
# =============================================================================
# 6b. Cross-Validation Stability Check
# =============================================================================

# Re-score each model's final settings with the same 5-fold splitter
# used for tuning, this time to see how much the score moves between
# folds.
# clone() gives an unfitted copy with identical settings, so the fitted
# pipelines above are left untouched, and the test set is never used.
cv_mcc_scores = {}
with warnings.catch_warnings():
    warnings.simplefilter('ignore', FutureWarning)
    for model_name, fitted_pipe in fitted_pipes.items():
        fold_scores = cross_val_score(
            clone(fitted_pipe),
            X_train,
            y_train,
            cv=cv_splitter,
            scoring='matthews_corrcoef',
        )
        cv_mcc_scores[model_name] = fold_scores
        print(
            f"{model_name}: {fold_scores.mean():.3f} "
            f"± {fold_scores.std():.3f} "
            f"({fold_scores.min():.3f} to {fold_scores.max():.3f})"
        )

Logistic Regression: 0.567 ± 0.023 (0.538 to 0.594)


Decision Tree: 0.589 ± 0.041 (0.526 to 0.634)


KNN: 0.568 ± 0.021 (0.530 to 0.586)


Naive Bayes: 0.519 ± 0.017 (0.499 to 0.545)


Support Vector Machine: 0.601 ± 0.022 (0.574 to 0.636)


In [17]:
# =============================================================================
# 6c. Evidence for the Observations
# =============================================================================

# --- What drives the Decision Tree ------------------------------------
# Feature names in the order the preprocessor produces them: the
# log-and-scale columns, the scaled bounded columns, then the one-hot
# names from the fitted encoder. get_feature_names_out() is called on
# the encoder itself, not on the whole preprocessor, because the log1p
# step has no such method and would make that call fail.
dt_pipe = fitted_pipes['Decision Tree']
cat_encoder = dt_pipe.named_steps['prep'].named_transformers_['cat']
dt_feature_names = (
    skewed_cols
    + bounded_cols
    + list(cat_encoder.get_feature_names_out(categorical_columns))
)
dt_importances = dt_pipe.named_steps['model'].feature_importances_
importance_table = pd.DataFrame({
    'feature': dt_feature_names,
    'importance': dt_importances.round(3),
}).sort_values('importance', ascending=False)

print("=== Top 5 Decision Tree Feature Importances ===")
print(importance_table.head(5).to_string(index=False))

# --- PageValues and buying ---------------------------------------------
has_page_value = shoppers_df['PageValues'] > 0
rate_with_page_value = shoppers_df.loc[has_page_value, 'Revenue'].mean()
rate_no_page_value = shoppers_df.loc[~has_page_value, 'Revenue'].mean()
share_with_page_value = has_page_value.mean()

print("\n=== PageValues and Purchase Rate ===")
print(f"Purchase rate when PageValues > 0: {rate_with_page_value:.3f}")
print(f"Purchase rate when PageValues = 0: {rate_no_page_value:.3f}")
print(f"Share of sessions with PageValues > 0: {share_with_page_value:.3f}")

# --- What Naive Bayes' independence assumption ignores ------------------
product_corr = shoppers_df['ProductRelated'].corr(
    shoppers_df['ProductRelated_Duration']
)
rate_corr = shoppers_df['BounceRates'].corr(shoppers_df['ExitRates'])

print("\n=== Correlations the Independence Assumption Ignores ===")
print(f"ProductRelated vs ProductRelated_Duration: {product_corr:.3f}")
print(f"BounceRates vs ExitRates: {rate_corr:.3f}")

# --- Logistic Regression vs KNN ---------------------------------------
# Their confusion counts are identical, so check whether they also label
# the same sessions, and count how many different scores KNN gives out
lr_labels = fitted_pipes['Logistic Regression'].predict(X_test)
knn_labels = fitted_pipes['KNN'].predict(X_test)
labels_differ = int((lr_labels != knn_labels).sum())
knn_scores = fitted_pipes['KNN'].predict_proba(X_test)[:, 1]

print("\n=== Logistic Regression vs KNN on the Test Set ===")
print(f"Sessions labelled differently: {labels_differ} of {len(X_test)}")
print(f"Distinct KNN probability values: {len(np.unique(knn_scores))}")

=== Top 5 Decision Tree Feature Importances ===
                feature  importance
             PageValues       0.782
            BounceRates       0.074
              Month_Nov       0.037
ProductRelated_Duration       0.032
         Administrative       0.023

=== PageValues and Purchase Rate ===
Purchase rate when PageValues > 0: 0.563
Purchase rate when PageValues = 0: 0.039
Share of sessions with PageValues > 0: 0.221

=== Correlations the Independence Assumption Ignores ===
ProductRelated vs ProductRelated_Duration: 0.861
BounceRates vs ExitRates: 0.913

=== Logistic Regression vs KNN on the Test Set ===
Sessions labelled differently: 110 of 2466
Distinct KNN probability values: 16


---
## 7. Observations

| ML Model Name | Observation about model performance |
|---|---|
| Logistic Regression | Logistic regression reaches a test MCC of 0.561 and the second-highest AUC, 0.911. It finds 208 of the 382 buyers in the test set and misses 174. Its training MCC of 0.586 is close to the test score, so it barely overfits, and that is with every setting left at its default. A straight-line model on log-scaled features keeps up with the more flexible models here, which suggests much of the signal in this data is close to linear. |
| Decision Tree | The decision tree gives the best test results of the five, with an MCC of 0.588, and it leads on every metric except recall. In its depth search the training MCC climbs to 1.000 as the tree grows, while the mean CV MCC peaks at 0.589 at depth 5 and falls to 0.476 with no limit, so capping the depth at 5 is what keeps it from overfitting. One feature, PageValues, carries 0.782 of its feature importance, so the tree is mostly one strong rule with small corrections. It is also the least stable model across folds, with the widest CV spread (standard deviation 0.041). |
| KNN | KNN settles on k = 15. The CV curve is almost flat from 15 to 51 neighbours (0.560 to 0.568), and k = 3 is the worst at 0.524. Its confusion counts are exactly the same as logistic regression's, but only by coincidence: the two label 110 of the 2,466 test sessions differently. Its AUC is lower, 0.896 against 0.911, because its probability is just the share of the 15 neighbours that are buyers, so it takes only 16 values and ranks sessions more coarsely than a continuous score. Its saved file is also far larger than the other four, because the model keeps every training row to find neighbours at prediction time. |
| Naive Bayes | With default settings Gaussian Naive Bayes collapses, predicting 2,149 of 2,466 test sessions as buyers, for an MCC of only 0.131. Tuning var_smoothing fixes most of that, but it is still the weakest model on MCC at 0.490. It has the highest recall of the five, 0.762 (291 of the 382 buyers), but the lowest precision, 0.454, with 350 false alarms. Part of the reason it trails is that its independence assumption clearly does not hold here: ProductRelated correlates with its own duration at 0.861, and BounceRates with ExitRates at 0.913, so the model counts the same evidence twice. |
| Support Vector Machine | The SVM's test MCC of 0.561 ties with logistic regression and KNN, but it has the largest train-test gap of the five: 0.658 on the training data against 0.561 on test. In 5-fold CV, though, it has the best mean MCC of all five, 0.601, and its test score is below even its weakest fold (0.574), which points to this test split being a harder draw for it rather than to a weak model. It was not grid-searched like the tree, KNN and Naive Bayes, and it is the slowest to fit, because `probability=True` makes it fit five extra internal models for Platt scaling. |
| Overall, Winner for your dataset? | My pick is the decision tree. I judge the models mainly on MCC, because only 15.5% of sessions are purchases and MCC only goes up when a model actually finds them. The tree leads on it, 0.588 against 0.561 for logistic regression, KNN and the SVM, and it also has the best F1, 0.643. The lead comes with a caveat: the SVM has the best 5-fold CV mean at 0.601, and the tree is the most variable model across folds (standard deviation 0.041), so on another test split the gap could shrink or reverse. There is also a caveat about PageValues itself. The purchase rate is 0.563 when it is above 0 and only 0.039 when it is 0, and Google Analytics derives page value from completed transactions, so in real-time use, before a session ends, this feature may already partly contain the answer the model is meant to predict. |

---
## 8. Saving the models

Each file holds the whole pipeline, the preprocessor and the model
together, so the app loads one file and predicts straight on raw
session rows. Because the preprocessor inside it is already
fitted, there is no separate encoding step and no retraining needed at
prediction time.

In [18]:
# =============================================================================
# 8a. Save the Fitted Pipelines
# =============================================================================

# Lowercase file names with no spaces, since the lab environment runs
# on Linux and is case-sensitive about paths.
model_file_paths = {
    'Logistic Regression': 'model/logistic_regression.joblib',
    'Decision Tree': 'model/decision_tree.joblib',
    'KNN': 'model/knn.joblib',
    'Naive Bayes': 'model/naive_bayes.joblib',
    'Support Vector Machine': 'model/svm.joblib',
}

print("=== Saved Model Files ===")
for model_name, file_path in model_file_paths.items():
    joblib.dump(fitted_pipes[model_name], file_path, compress=3)
    file_size_kb = os.path.getsize(file_path) / 1024
    print(f"{file_path}: {file_size_kb:.1f} KB")

print("\n=== Library Versions ===")
print(f"scikit-learn: {sklearn.__version__}")
print(f"pandas:       {pd.__version__}")
print(f"numpy:        {np.__version__}")
print(f"joblib:       {joblib.__version__}")

=== Saved Model Files ===
model/logistic_regression.joblib: 2.9 KB
model/decision_tree.joblib: 4.6 KB
model/knn.joblib: 563.0 KB
model/naive_bayes.joblib: 4.2 KB
model/svm.joblib: 186.6 KB

=== Library Versions ===
scikit-learn: 1.9.1
pandas:       3.1.0
numpy:        2.5.4
joblib:       1.6.0


In [19]:
# =============================================================================
# 8b. Verify the Saved Models
# =============================================================================

# Read test_data.csv back from disk, the way the app will, and split the
# target off. The metrics check uses this copy rather than X_test, which
# is only used to compare each reloaded model with its in-memory version.
reloaded_test_data = pd.read_csv('test_data.csv')
X_test_from_csv = reloaded_test_data.drop(columns=['Revenue'])
y_test_from_csv = reloaded_test_data['Revenue']

verification_checks = {}
for model_name, file_path in model_file_paths.items():
    reloaded_pipe = joblib.load(file_path)

    same_predictions = np.array_equal(
        reloaded_pipe.predict(X_test),
        fitted_pipes[model_name].predict(X_test),
    )

    reloaded_metrics = compute_metrics(
        reloaded_pipe, X_test_from_csv, y_test_from_csv
    )
    same_metrics = reloaded_metrics == test_scores[model_name]

    rows_scored = len(y_test_from_csv)
    scored_on_2466_rows = rows_scored == 2466

    tn, fp, fn, tp = confusion_matrix(
        y_test_from_csv, reloaded_pipe.predict(X_test_from_csv)
    ).ravel()
    accuracy_from_confusion = round((tp + tn) / rows_scored, 3)
    accuracy_matches_confusion = (
        accuracy_from_confusion == reloaded_metrics['Accuracy']
    )

    verification_checks[model_name] = {
        'same predictions': same_predictions,
        'same metrics': same_metrics,
        '2466 rows': scored_on_2466_rows,
        'accuracy ok': accuracy_matches_confusion,
    }

verification_table = pd.DataFrame(verification_checks).T
print("=== Verification Checks (should all be True) ===")
print(verification_table)
print(f"\nEvery check passes: {verification_table.all(axis=None)}")

=== Verification Checks (should all be True) ===
                        same predictions  same metrics  2466 rows  accuracy ok
Logistic Regression                 True          True       True         True
Decision Tree                       True          True       True         True
KNN                                 True          True       True         True
Naive Bayes                         True          True       True         True
Support Vector Machine              True          True       True         True

Every check passes: True
